In [1]:
import sys
from pathlib import Path
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Find project root and add to path
def find_project_root(marker="data/imagenet100"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find project root containing {marker}")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import ImageNet100Dataset, IMAGENET_PREPROCESS, collect_files, load_labels

DATA_ROOT = PROJECT_ROOT / "data" / "imagenet100"
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Embeddings will be saved to: {EMBEDDINGS_DIR}")
print(f"Device: {device}")

Project root: /Users/ayushkumarsingh/ABNS-Project/tversky-asymmetric
Embeddings will be saved to: /Users/ayushkumarsingh/ABNS-Project/tversky-asymmetric/data/embeddings
Device: mps


In [2]:
import json
from sklearn.model_selection import train_test_split

# Load labels
labels = load_labels(DATA_ROOT)

# Collect files
train_folders = [DATA_ROOT / f"train.X{i}" for i in [1, 2, 3, 4]]
train_items_full = collect_files(train_folders)
heldout_items = collect_files([DATA_ROOT / "val.X"])

# Build the synset -> idx mapping
all_synsets = sorted(set(synset for _, synset in train_items_full))
synset_to_idx = {synset: i for i, synset in enumerate(all_synsets)}

# Re-do the 90/10 stratified split with the same random_state
train_labels_for_split = [synset_to_idx[s] for _, s in train_items_full]
train_subset, val_subset, _, _ = train_test_split(
    train_items_full,
    train_labels_for_split,
    test_size=0.10,
    stratify=train_labels_for_split,
    random_state=42,
)

print(f"Train:    {len(train_subset):>7,}")
print(f"Val:      {len(val_subset):>7,}")
print(f"Held-out: {len(heldout_items):>7,}")

Train:    117,000
Val:       13,000
Held-out:   5,000


In [3]:
print("Loading Barlow Twins backbone...")
backbone = torch.hub.load('facebookresearch/barlowtwins:main', 'resnet50')
backbone.fc = nn.Identity()
for p in backbone.parameters():
    p.requires_grad = False
backbone.eval()
backbone = backbone.to(device)
print("Backbone ready.")

Loading Barlow Twins backbone...


Using cache found in /Users/ayushkumarsingh/.cache/torch/hub/facebookresearch_barlowtwins_main
/opt/homebrew/Caskroom/miniconda/base/envs/tversky/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/tversky/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Backbone ready.


In [4]:
@torch.no_grad()
def extract_embeddings(dataset, backbone, device, batch_size=256, num_workers=4):
    """
    Run every image in the dataset through the backbone and collect embeddings.
    
    Returns:
        features: tensor of shape (N, 2048) on CPU
        labels: tensor of shape (N,) on CPU
    """
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        persistent_workers=True,
    )
    
    all_features = []
    all_labels = []
    
    n_batches = len(loader)
    start = time.time()
    
    for i, (images, labels) in enumerate(loader):
        images = images.to(device)
        features = backbone(images)
        # Move to CPU immediately to free MPS memory
        all_features.append(features.cpu())
        all_labels.append(labels)
        
        # Progress indicator every 50 batches
        if (i + 1) % 50 == 0 or i == 0:
            elapsed = time.time() - start
            rate = (i + 1) * batch_size / elapsed
            eta = (n_batches - i - 1) * batch_size / rate
            print(f"  Batch {i+1:4d}/{n_batches} | "
                  f"{rate:.0f} img/s | "
                  f"elapsed {elapsed:.0f}s | "
                  f"ETA {eta:.0f}s")
    
    # Concatenate everything into single tensors
    features = torch.cat(all_features, dim=0)
    labels = torch.cat(all_labels, dim=0)
    
    elapsed = time.time() - start
    print(f"  Done. {len(features):,} embeddings in {elapsed:.1f}s "
          f"({len(features)/elapsed:.0f} img/s)")
    
    return features, labels


print("Function defined.")

Function defined.


In [5]:
# Build the datasets
train_dataset = ImageNet100Dataset(train_subset, synset_to_idx, transform=IMAGENET_PREPROCESS)
val_dataset = ImageNet100Dataset(val_subset, synset_to_idx, transform=IMAGENET_PREPROCESS)
heldout_dataset = ImageNet100Dataset(heldout_items, synset_to_idx, transform=IMAGENET_PREPROCESS)

# Held-out (5K, smallest — should take ~30s)
print("=" * 60)
print("HELD-OUT SET")
print("=" * 60)
heldout_features, heldout_labels = extract_embeddings(heldout_dataset, backbone, device)

# Save
torch.save(
    {"features": heldout_features, "labels": heldout_labels},
    EMBEDDINGS_DIR / "heldout_embeddings.pt",
)
print(f"  Saved to {EMBEDDINGS_DIR / 'heldout_embeddings.pt'}")
print(f"  Features shape: {heldout_features.shape}, dtype: {heldout_features.dtype}")
print(f"  Labels shape: {heldout_labels.shape}, dtype: {heldout_labels.dtype}")

HELD-OUT SET
  Batch    1/20 | 30 img/s | elapsed 8s | ETA 161s
  Done. 5,000 embeddings in 41.3s (121 img/s)
  Saved to /Users/ayushkumarsingh/ABNS-Project/tversky-asymmetric/data/embeddings/heldout_embeddings.pt
  Features shape: torch.Size([5000, 2048]), dtype: torch.float32
  Labels shape: torch.Size([5000]), dtype: torch.int64


In [6]:
# Load the just-saved file and verify it round-trips correctly
loaded = torch.load(EMBEDDINGS_DIR / "heldout_embeddings.pt", weights_only=True)
print(f"Loaded keys: {list(loaded.keys())}")
print(f"Features: {loaded['features'].shape}, {loaded['features'].dtype}")
print(f"Labels:   {loaded['labels'].shape}, {loaded['labels'].dtype}")

# Quick stats to confirm embeddings are sensible
features = loaded['features']
print(f"\nFeature stats:")
print(f"  Mean: {features.mean():.4f}")
print(f"  Std:  {features.std():.4f}")
print(f"  Min:  {features.min():.4f}")
print(f"  Max:  {features.max():.4f}")
print(f"  Fraction of zero values: {(features == 0).float().mean():.3f}")

# File size on disk
file_size = (EMBEDDINGS_DIR / "heldout_embeddings.pt").stat().st_size / (1024 * 1024)
print(f"\nFile size on disk: {file_size:.1f} MB")

Loaded keys: ['features', 'labels']
Features: torch.Size([5000, 2048]), torch.float32
Labels:   torch.Size([5000]), torch.int64

Feature stats:
  Mean: 0.0523
  Std:  0.1080
  Min:  0.0000
  Max:  5.1079
  Fraction of zero values: 0.382

File size on disk: 39.1 MB


In [7]:
print("=" * 60)
print("VALIDATION SET")
print("=" * 60)
val_features, val_labels = extract_embeddings(val_dataset, backbone, device)

torch.save(
    {"features": val_features, "labels": val_labels},
    EMBEDDINGS_DIR / "val_embeddings.pt",
)
print(f"  Saved to {EMBEDDINGS_DIR / 'val_embeddings.pt'}")
print(f"  Features shape: {val_features.shape}")

VALIDATION SET
  Batch    1/51 | 33 img/s | elapsed 8s | ETA 392s
  Batch   50/51 | 134 img/s | elapsed 96s | ETA 2s
  Done. 13,000 embeddings in 97.3s (134 img/s)
  Saved to /Users/ayushkumarsingh/ABNS-Project/tversky-asymmetric/data/embeddings/val_embeddings.pt
  Features shape: torch.Size([13000, 2048])


In [8]:
print("=" * 60)
print("TRAIN SET (117K images, this will take ~15 minutes)")
print("=" * 60)
train_features, train_labels = extract_embeddings(train_dataset, backbone, device)

torch.save(
    {"features": train_features, "labels": train_labels},
    EMBEDDINGS_DIR / "train_embeddings.pt",
)
print(f"  Saved to {EMBEDDINGS_DIR / 'train_embeddings.pt'}")
print(f"  Features shape: {train_features.shape}")

TRAIN SET (117K images, this will take ~15 minutes)
  Batch    1/458 | 26 img/s | elapsed 10s | ETA 4428s
  Batch   50/458 | 131 img/s | elapsed 98s | ETA 799s
  Batch  100/458 | 135 img/s | elapsed 190s | ETA 680s
  Batch  150/458 | 135 img/s | elapsed 285s | ETA 584s
  Batch  200/458 | 135 img/s | elapsed 380s | ETA 491s
  Batch  250/458 | 132 img/s | elapsed 484s | ETA 403s
  Batch  300/458 | 131 img/s | elapsed 587s | ETA 309s
  Batch  350/458 | 130 img/s | elapsed 691s | ETA 213s
  Batch  400/458 | 129 img/s | elapsed 796s | ETA 115s
  Batch  450/458 | 128 img/s | elapsed 903s | ETA 16s
  Done. 117,000 embeddings in 918.3s (127 img/s)
  Saved to /Users/ayushkumarsingh/ABNS-Project/tversky-asymmetric/data/embeddings/train_embeddings.pt
  Features shape: torch.Size([117000, 2048])


In [9]:
# Verify all three saved files
for split_name in ["train", "val", "heldout"]:
    path = EMBEDDINGS_DIR / f"{split_name}_embeddings.pt"
    loaded = torch.load(path, weights_only=True)
    features = loaded["features"]
    labels = loaded["labels"]
    file_mb = path.stat().st_size / (1024 * 1024)
    
    # Verify all classes are present
    n_classes_present = len(torch.unique(labels))
    
    print(f"{split_name:>8}: features={tuple(features.shape)} "
          f"labels={tuple(labels.shape)} "
          f"classes_present={n_classes_present} "
          f"file_size={file_mb:.0f}MB")

# Total disk usage
total_mb = sum(
    (EMBEDDINGS_DIR / f"{s}_embeddings.pt").stat().st_size / (1024 * 1024)
    for s in ["train", "val", "heldout"]
)
print(f"\nTotal disk usage: {total_mb:.0f} MB")

   train: features=(117000, 2048) labels=(117000,) classes_present=100 file_size=915MB
     val: features=(13000, 2048) labels=(13000,) classes_present=100 file_size=102MB
 heldout: features=(5000, 2048) labels=(5000,) classes_present=100 file_size=39MB

Total disk usage: 1056 MB


In [10]:
del backbone
import gc
gc.collect()
if device.type == "mps":
    torch.mps.empty_cache()

print("Backbone released.")

Backbone released.


In [12]:
import json

# Reload the labels dictionary (it was shadowed by the loop variable earlier)
synset_to_name = load_labels(DATA_ROOT)

mapping = {
    "synset_to_idx": synset_to_idx,
    "idx_to_synset": {i: s for s, i in synset_to_idx.items()},
    "idx_to_name": {i: synset_to_name[s] for s, i in synset_to_idx.items()},
}

with open(EMBEDDINGS_DIR / "label_mapping.json", "w") as f:
    json.dump(mapping, f, indent=2)

print(f"Label mapping saved to {EMBEDDINGS_DIR / 'label_mapping.json'}")
print(f"\nFirst 3 entries:")
for i in range(3):
    print(f"  {i} -> {mapping['idx_to_synset'][i]} -> {mapping['idx_to_name'][i]}")

Label mapping saved to /Users/ayushkumarsingh/ABNS-Project/tversky-asymmetric/data/embeddings/label_mapping.json

First 3 entries:
  0 -> n01440764 -> tench, Tinca tinca
  1 -> n01443537 -> goldfish, Carassius auratus
  2 -> n01484850 -> great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias
